# Week 4 — CNN-LSTM Hybrid (Video-Level Deepfake Detection)

End-to-end training: ResNet-50 (ImageNet init) + BiLSTM, jointly fine-tuned on 32-frame clips.

**Runtime:** GPU recommended (Colab T4 ~30-50 min for 20 epochs on 700 train videos).

**Inputs:** expects the project tree present at `PROJECT_ROOT` (default: Google Drive).

## 1. Environment setup

If on Colab, mount Drive and point `PROJECT_ROOT` at the synced repo. If running locally, set `PROJECT_ROOT` to the repo path and skip the Drive mount.

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('In Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/deepfake-detection')
else:
    PROJECT_ROOT = Path(r'F:\ml_project\deepfake-detection')

assert PROJECT_ROOT.exists(), f'PROJECT_ROOT not found: {PROJECT_ROOT}'
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# Install only what Colab is missing (PyTorch is preinstalled).
if IN_COLAB:
    !pip install -q tqdm scikit-learn matplotlib pandas Pillow

## 2. Imports & seed

In [ ]:
import json, random
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.data.video_dataset import build_video_dataloaders, VideoSequenceDataset
from src.models.hybrid_model import create_hybrid_model_and_optimizer, HybridCNNLSTM

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Hyperparameters

Tune `BATCH_SIZE` to GPU memory. T4 (16GB) handles `BATCH_SIZE=4` comfortably with 32 frames at 224×224 end-to-end; raise it if you have an A100.

In [ ]:
CONFIG = {
    'batch_size': 4,
    'epochs': 20,
    'num_workers': 2,
    'num_frames': 32,
    'image_size': 224,
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'dropout': 0.5,
    'lstm_hidden_size': 256,
    'lstm_num_layers': 2,
    'bidirectional': True,
    'freeze_backbone': False,
    'trainable_backbone_layers': 4,
    'pretrained': True,
    'early_stop_patience': 5,
}
CONFIG

## 4. Data loaders

In [ ]:
train_loader, val_loader, test_loader = build_video_dataloaders(
    project_root=PROJECT_ROOT,
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
    num_frames=CONFIG['num_frames'],
    image_size=CONFIG['image_size'],
)
print(f"Train videos: {len(train_loader.dataset)} | class counts: {train_loader.dataset.class_counts}")
print(f"Val   videos: {len(val_loader.dataset)} | class counts: {val_loader.dataset.class_counts}")
print(f"Test  videos: {len(test_loader.dataset)} | class counts: {test_loader.dataset.class_counts}")

# Smoke test: one batch shape
clip, label = next(iter(val_loader))
print('Batch clip shape:', tuple(clip.shape), '| label shape:', tuple(label.shape))

## 5. Model

In [ ]:
model, criterion, optimizer = create_hybrid_model_and_optimizer(
    device=DEVICE,
    pretrained=CONFIG['pretrained'],
    freeze_backbone=CONFIG['freeze_backbone'],
    trainable_backbone_layers=CONFIG['trainable_backbone_layers'],
    lstm_hidden_size=CONFIG['lstm_hidden_size'],
    lstm_num_layers=CONFIG['lstm_num_layers'],
    bidirectional=CONFIG['bidirectional'],
    dropout=CONFIG['dropout'],
    learning_rate=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')

## 6. Training loop

In [ ]:
MODELS_DIR  = PROJECT_ROOT / 'models'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = MODELS_DIR / 'hybrid_best.pth'
HISTORY_PATH    = OUTPUTS_DIR / 'train_history_hybrid.json'
CURVE_PATH      = OUTPUTS_DIR / 'hybrid_training_curves.png'

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
best_val_acc = 0.0
patience_counter = 0

for epoch in range(1, CONFIG['epochs'] + 1):
    # Train
    tl = ta = 0.0
    n = 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch} [train]', leave=False):
        m = model.train_step(batch, criterion, optimizer, DEVICE)
        tl += m['loss']; ta += m['accuracy']; n += 1
    train_loss, train_acc = tl / max(n, 1), ta / max(n, 1)

    # Validate
    vl = va = 0.0
    n = 0
    for batch in tqdm(val_loader, desc=f'Epoch {epoch} [val]', leave=False):
        m = model.val_step(batch, criterion, DEVICE)
        vl += m['loss']; va += m['accuracy']; n += 1
    val_loss, val_acc = vl / max(n, 1), va / max(n, 1)

    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(lr)

    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
          f'val_loss={val_loss:.4f} val_acc={val_acc:.4f} | lr={lr:.6f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_acc': best_val_acc,
            'history': history,
            'config': CONFIG,
        }, CHECKPOINT_PATH)
        print(f'  Saved checkpoint: {CHECKPOINT_PATH}')
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['early_stop_patience']:
            print(f'Early stopping at epoch {epoch} (no val_acc improvement for {patience_counter} epochs)')
            break

with HISTORY_PATH.open('w', encoding='utf-8') as fp:
    json.dump(history, fp, indent=2)
print(f'Best val acc: {best_val_acc:.4f}')

## 7. Training curves

In [ ]:
epochs_x = range(1, len(history['train_loss']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs_x, history['train_loss'], label='Train')
ax[0].plot(epochs_x, history['val_loss'],   label='Val')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss'); ax[0].set_title('Hybrid Loss'); ax[0].legend()
ax[1].plot(epochs_x, history['train_acc'], label='Train')
ax[1].plot(epochs_x, history['val_acc'],   label='Val')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Accuracy'); ax[1].set_title('Hybrid Accuracy'); ax[1].legend()
plt.tight_layout()
plt.savefig(CURVE_PATH, dpi=150, bbox_inches='tight')
plt.show()

## 8. Test-set evaluation

Loads the best checkpoint and reports video-level metrics.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              roc_auc_score)

ck = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ck['model_state_dict'])
model.eval()

y_true, y_pred, y_prob = [], [], []
with torch.no_grad():
    for clips, labels in tqdm(test_loader, desc='Test'):
        clips = clips.to(DEVICE)
        labels = labels.to(DEVICE).float().view(-1)
        probs = torch.sigmoid(model(clips))
        y_true.extend(labels.long().cpu().tolist())
        y_prob.extend(probs.cpu().tolist())
        y_pred.extend((probs >= 0.5).long().cpu().tolist())

y_true = np.array(y_true); y_pred = np.array(y_pred); y_prob = np.array(y_prob)
print(f'Accuracy : {accuracy_score(y_true, y_pred):.4f}')
print(f'Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}')
print(f'Recall   : {recall_score(y_true, y_pred, zero_division=0):.4f}')
print(f'F1-Score : {f1_score(y_true, y_pred, zero_division=0):.4f}')
try:
    print(f'ROC-AUC  : {roc_auc_score(y_true, y_prob):.4f}')
except ValueError:
    pass
print()
print(classification_report(y_true, y_pred, target_names=['real', 'fake'], digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Hybrid - Confusion Matrix'); plt.colorbar()
ticks = np.arange(2)
plt.xticks(ticks, ['Pred Real', 'Pred Fake']); plt.yticks(ticks, ['True Real', 'True Fake'])
thr = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'), ha='center', va='center',
                 color='white' if cm[i, j] > thr else 'black')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'hybrid_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Download checkpoint (Colab only)

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(CHECKPOINT_PATH))